# Figure: SVGD Stein-update scaling

Loads `results/svgd/scaling.json` (`bench/svgd/scaling.py`). Never recomputes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

_here = Path.cwd()
_candidates = [_here / "results" / "svgd", _here.parents[1] / "results" / "svgd"]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
BASELINE = "#8C8C8C"


In [ ]:
data = json.loads((RESULTS / "scaling.json").read_text())
meta = data["metadata"]; records = data["records"]
ns = [r["n"] for r in records]
phi = [r["phi"]["min_s"] * 1e3 for r in records]
vg = [r["value_and_grad"]["min_s"] * 1e3 for r in records]
exact = [(r["n"], r["exact_s"] * 1e3) for r in records if "exact_s" in r]
print(f"device={meta['device_kind']} jax={meta['jax_version']}")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
ax.plot(ns, phi, "o-", color=PALETTE[0], label="Stein update (forward)")
ax.plot(ns, vg, "s-", color=PALETTE[1], label="Stein update (fwd+grad)")
if exact:
    ax.plot([e[0] for e in exact], [e[1] for e in exact], "^--", color=BASELINE,
            label="exact O(N^2)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("N particles"); ax.set_ylabel("time [ms]")
ax.set_title(f"Tree-accelerated SVGD update scaling ({meta['device_kind']})")
ax.legend()
out = FIGDIR / "fig_svgd_scaling.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)
